# Notebook 1 – Data Preprocessing Fundamentals

## 1. What is Data Preprocessing?

**Concept:** Data preprocessing is the process of checking, cleaning, and transforming raw data before using it for analysis or Machine Learning. It can include fixing data types, handling missing values, removing unintended duplicates, validating values, encoding categories, and scaling numerical features.

**Example:** A customer dataset may contain blank `TotalCharges`, inconsistent category labels, and numeric values stored as text.

**Why it is required:** A model trained on poor-quality or incorrectly represented data may learn misleading patterns or fail during training or prediction.


In [1]:
import pandas as pd
import numpy as np

sample = pd.DataFrame({
    "customerID": ["C001", "C002", "C003", "C004"],
    "tenure": [12, 24, np.nan, 8],
    "MonthlyCharges": [70.5, 85.0, 60.0, 75.5],
    "TotalCharges": ["846.0", "2040.0", " ", "604.0"],
    "Contract": ["Month-to-month", "One year", "Two year", "Month-to-month"],
    "Churn": ["No", "No", "Yes", "No"]
})

sample

,customerID,tenure,MonthlyCharges,TotalCharges,Contract,Churn
0,C001,12.0,70.5,846.0,Month-to-month,No
1,C002,24.0,85.0,2040.0,One year,No
2,C003,NaN,60.0,,Two year,Yes
3,C004,8.0,75.5,604.0,Month-to-month,No


**Output explanation:** The sample contains a missing `tenure` value and a blank `TotalCharges` entry stored as text. These issues should be investigated before modeling; we should not automatically assume one cleaning technique is correct for every column.

## 2. Why is Data Preprocessing Required?

**Concept:** Preprocessing helps make data suitable for analysis and model training. It can improve reliability, consistency, and compatibility with algorithms.

**Example / business use case:** A telecom company predicting customer churn needs valid customer attributes. If charges are stored as text or key fields are missing, the model may not process the data correctly or may produce unreliable results.

**Typical problems to check:**
- Missing values
- Duplicate records
- Incorrect data types
- Inconsistent labels or formats
- Invalid values and impossible ranges
- Outliers that may be errors or valid rare cases
- Irrelevant features
- Data leakage between training and evaluation


In [2]:
# Inspect common data-quality indicators in the sample
print("Shape:", sample.shape)
print("\nData types:")
print(sample.dtypes)
print("\nMissing values:")
print(sample.isna().sum())
print("\nExact duplicate rows:", sample.duplicated().sum())
print("\nBlank/whitespace-only TotalCharges:",
      sample["TotalCharges"].astype("string").str.strip().eq("").sum())

Shape: (4, 6)

Data types:
customerID            str
tenure            float64
MonthlyCharges    float64
TotalCharges          str
Contract              str
Churn                 str
dtype: object

Missing values:
customerID        0
tenure            1
MonthlyCharges    0
TotalCharges      0
Contract          0
Churn             0
dtype: int64

Exact duplicate rows: 0

Blank/whitespace-only TotalCharges: 1


**Output explanation:** The checks show the dataset dimensions, each column's data type, null counts, exact duplicate count, and a separate count of blank strings. `isna()` does not count a blank string as a missing value, so both checks can be necessary.

## 3. Raw Data vs Clean Data

**Concept:** Raw data is the data as received. Clean data has been reviewed and appropriate quality issues have been addressed. “Clean” does not mean changing every unusual value; each decision should be justified and documented.

**Example:** Convert a blank charge to a true missing value, convert valid charge strings to numbers, and then decide how to handle missing charges based on the data and business context.

In [3]:
cleaning_demo = sample.copy()

# Standardize whitespace-only values as missing
cleaning_demo = cleaning_demo.replace(r"^\s*$", np.nan, regex=True)

# Convert TotalCharges to a numeric type; invalid values become NaN
cleaning_demo["TotalCharges"] = pd.to_numeric(
    cleaning_demo["TotalCharges"], errors="coerce"
)

print("Before:")
print(sample[["TotalCharges"]])
print("\nAfter standardizing blanks and converting to numeric:")
print(cleaning_demo[["TotalCharges"]])
print("\nAfter data types:")
print(cleaning_demo.dtypes[["tenure", "TotalCharges"]])

Before:
  TotalCharges
0        846.0
1       2040.0
2             
3        604.0

After standardizing blanks and converting to numeric:
   TotalCharges
0         846.0
1        2040.0
2           NaN
3         604.0

After data types:
tenure          float64
TotalCharges    float64
dtype: object


**Output explanation:** `TotalCharges` becomes numeric, and the blank entry becomes `NaN`. The missing value has been identified, not imputed. Choosing how to fill or retain it is a separate decision that should consider the column meaning and the modeling workflow.

## 4. Data Quality

**Concept:** Data quality describes how suitable the data is for its intended use. It includes completeness, consistency, accuracy, validity, and integrity.

**Example:** A customer age of `-5` may violate a valid range; a missing customer ID may make it difficult to connect a record to the correct customer.

**Important:** A value that looks unusual is not automatically wrong. Verify it using business rules, source systems, and domain knowledge where possible.

## 5. Data Consistency

**Concept:** Consistency means the same information is represented in a uniform way across records and fields.

**Example:** `Yes`, `YES`, and ` yes ` may all be intended to represent the same category.

**Why it matters:** Inconsistent labels can be treated as different categories by an encoder or analysis.

In [4]:
categories_demo = pd.Series(["Yes", "YES", " yes ", "No", "No "], name="Churn")
print("Before standardization:")
print(categories_demo.value_counts(dropna=False))

standardized_categories = categories_demo.astype("string").str.strip().str.lower()
print("\nAfter standardization:")
print(standardized_categories.value_counts(dropna=False))

Before standardization:
Churn
Yes      1
YES      1
 yes     1
No       1
No       1
Name: count, dtype: int64

After standardization:
Churn
yes    3
no     2
Name: count, dtype: int64[pyarrow]


**Output explanation:** Trimming whitespace and standardizing case combines equivalent spellings. Apply such standardization only when those values genuinely have the same meaning; preserve meaningful distinctions.

## 6. Data Completeness

**Concept:** Completeness refers to whether required information is present.

**Example:** A customer record may have no value for `tenure` or `TotalCharges`.

**Why it matters:** Some algorithms cannot accept missing values directly. Missingness can also carry information, so investigate why values are absent before choosing an approach.

In [5]:
completeness = pd.DataFrame({
    "missing_count": cleaning_demo.isna().sum(),
    "missing_percent": (cleaning_demo.isna().mean() * 100).round(2)
})
completeness

,missing_count,missing_percent
customerID,0,0.0
tenure,1,25.0
MonthlyCharges,0,0.0
TotalCharges,1,25.0
Contract,0,0.0
Churn,0,0.0


**Output explanation:** The table reports the number and percentage of missing values in each column. This helps prioritize investigation; it does not by itself determine whether to drop, impute, or retain a field.

## 7. Data Accuracy

**Concept:** Accuracy means values correctly represent the real-world facts they are supposed to record.

**Example:** If a customer's actual monthly charge is ₹700 but the record says ₹7,000 because of a typing error, the value is inaccurate.

**How to assess it:** Compare against trusted source records, business rules, or reconciliation checks. Accuracy cannot always be confirmed from the dataset alone.

## 8. Data Validity

**Concept:** Validity means values follow expected rules, formats, types, and allowed ranges.

**Example:** A percentage should normally be between 0 and 100, and a charge should not be negative if the business definition does not permit negative charges.

In [6]:
validation_demo = pd.DataFrame({
    "Age": [25, -3, 42, 150],
    "DiscountPercent": [10, 105, 0, 25],
    "MonthlyCharges": [70.0, 80.0, -5.0, 90.0]
})

checks = pd.DataFrame({
    "age_nonnegative": validation_demo["Age"] >= 0,
    "age_within_example_limit_120": validation_demo["Age"].between(0, 120),
    "discount_0_to_100": validation_demo["DiscountPercent"].between(0, 100),
    "charges_nonnegative": validation_demo["MonthlyCharges"] >= 0
})
print("Input:")
display(validation_demo)
print("Validation results (True means the rule passes):")
display(checks)

Input:


,Age,DiscountPercent,MonthlyCharges
0,25,10,70.0
1,-3,105,80.0
2,42,0,-5.0
3,150,25,90.0


Validation results (True means the rule passes):


,age_nonnegative,age_within_example_limit_120,discount_0_to_100,charges_nonnegative
0,True,True,True,True
1,False,False,False,True
2,True,True,True,False
3,True,False,True,True


**Output explanation:** Rows that return `False` violate the example rules. The age limit of 120 is only an illustrative rule; real validation limits must be defined for the actual business domain.

## 9. Data Integrity

**Concept:** Integrity means data remains correct and relationships between records and fields are maintained.

**Example:** An invoice line should refer to an invoice that exists. A customer ID used to join two tables should identify the intended customer.

**Why it matters:** Broken relationships can create missing joins, incorrect totals, or duplicated records during analysis.

In [7]:
customers = pd.DataFrame({"customerID": ["C001", "C002", "C003"]})
subscriptions = pd.DataFrame({
    "customerID": ["C001", "C002", "C999"],
    "plan": ["Basic", "Premium", "Basic"]
})

integrity_check = subscriptions.loc[
    ~subscriptions["customerID"].isin(customers["customerID"])
]
print("Subscription records with no matching customer:")
display(integrity_check)

Subscription records with no matching customer:


,customerID,plan
2,C999,Basic


**Output explanation:** `C999` appears in subscriptions but not in the customer table. This is a referential-integrity issue to investigate before joining the tables.

## 10. Data Leakage

**Concept:** Data leakage happens when information unavailable at prediction time, or information from validation/test data, influences model training or preprocessing.

**Example:** Calculating a scaler using the full dataset before the train/test split lets information from the test set influence preprocessing.

**Correct principle:** Split the data first. Fit preprocessing steps using training data only, then use the fitted transformations on validation/test data. A Scikit-learn `Pipeline` helps enforce this pattern.

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

leakage_demo = pd.DataFrame({
    "feature": [10, 12, 11, 13, 1000, 9, 12, 11],
    "target":  [0, 0, 0, 1, 1, 0, 1, 0]
})

X = leakage_demo[["feature"]]
y = leakage_demo["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit only on training data
X_test_scaled = scaler.transform(X_test)         # transform test data only

print("Training mean used by scaler:", round(float(scaler.mean_[0]), 2))
print("Raw test values:", X_test["feature"].tolist())
print("Scaled test values:", X_test_scaled.ravel().round(2).tolist())

Training mean used by scaler: 176.0
Raw test values: [9, 13]
Scaled test values: [-0.45, -0.44]


**Output explanation:** The scaler learns its mean and standard deviation from `X_train` only. The test values are transformed with those training statistics, keeping the test set separate from fitting. The exact split is reproducible because `random_state=42` is set.

## 11. Training Data, Validation Data, and Test Data

- **Training data:** used to fit the model and fit preprocessing transformations.
- **Validation data:** used to compare approaches and tune model choices.
- **Test data:** held back for final, unbiased evaluation after decisions are complete.

**Example:** For churn prediction, train on one portion of historical customer records, use validation data to choose settings, and use test data only for the final evaluation.

**Important:** For time-dependent problems, a chronological split may be more appropriate than a random split.

In [9]:
from sklearn.model_selection import train_test_split

# First separate the final test set
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Split the remaining development data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=0.25, random_state=42, stratify=y_dev
)

print("Train rows:", len(X_train))
print("Validation rows:", len(X_val))
print("Test rows:", len(X_test))
print("Total rows:", len(X_train) + len(X_val) + len(X_test))

Train rows: 4
Validation rows: 2
Test rows: 2
Total rows: 8


**Output explanation:** This creates approximately a 60% training, 20% validation, and 20% test split. Stratification attempts to preserve class proportions. On very small datasets, stratification may fail if a class has too few examples; use a suitable split strategy for the real dataset.

## 12. Preprocessing Pipeline

**Concept:** A preprocessing pipeline organizes transformations into a repeatable sequence. It reduces the chance of applying steps inconsistently and helps prevent leakage when fitted correctly.

**Typical workflow:**

**Identify → Analyze → Clean → Transform → Validate → Prepare for Machine Learning**

1. Load and inspect the data.
2. Document the data-quality problems.
3. Split features and target, then split training/validation/test data.
4. Fit imputation, encoding, scaling, and feature-selection steps on training data only.
5. Transform validation/test data with the already-fitted steps.
6. Validate outputs and document decisions.
7. Evaluate the model on held-out test data only after model selection is complete.

In [10]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Fit only on the training split
numeric_pipeline.fit(X_train)

X_train_ready = numeric_pipeline.transform(X_train)
X_val_ready = numeric_pipeline.transform(X_val)
X_test_ready = numeric_pipeline.transform(X_test)

print("Train transformed shape:", X_train_ready.shape)
print("Validation transformed shape:", X_val_ready.shape)
print("Test transformed shape:", X_test_ready.shape)

Train transformed shape: (4, 1)
Validation transformed shape: (2, 1)
Test transformed shape: (2, 1)


**Output explanation:** The pipeline learns the median imputation value and scaling statistics from training data. The same fitted transformations are then applied to validation and test data. This example has one numerical feature; later notebooks will extend the approach to mixed numerical and categorical columns.

## 13. Load the Sprint 4 Telco Dataset (Optional)

Use this section to connect the concepts to the IBM Telco Customer Churn dataset from Sprint 4. Place `WA_Fn-UseC_-Telco-Customer-Churn.csv` in the notebook's working folder, or update `DATA_PATH` to its location.

In [11]:
from pathlib import Path

DATA_PATH = Path("WA_Fn-UseC_-Telco-Customer-Churn.csv")

if DATA_PATH.exists():
    telco_df = pd.read_csv(DATA_PATH)
    print("Dataset loaded successfully.")
    print("Shape:", telco_df.shape)
    display(telco_df.head())
    print("\nData types:")
    display(telco_df.dtypes)
    print("\nMissing values:")
    display(telco_df.isna().sum().sort_values(ascending=False).head(15))
    print("\nExact duplicate rows:", telco_df.duplicated().sum())
else:
    print(f"Dataset not found at: {DATA_PATH.resolve()}")
    print("Place the CSV in the notebook folder or update DATA_PATH.")

Dataset not found at: C:\Users\ramki\sprint_5\WA_Fn-UseC_-Telco-Customer-Churn.csv
Place the CSV in the notebook folder or update DATA_PATH.


**Output explanation:** If the CSV is available, this cell displays its shape, sample rows, data types, missing-value counts, and exact duplicate count. If it is not found, the notebook reports the expected path without stopping the earlier practice examples.